In [1]:
class SortedSet(set):

    def __repr__(self):
        items = ", ".join(f"'{x}'" for x in sorted(self))
        return f"{{{items}}}"


def Sym(n):
    return ("sym", str(n))


def Not(a):
    return ("not", a)


def And(a, b):
    return ("and", a, b)


def Or(a, b):
    return ("or", a, b)


def Imp(a, b):
    return ("imp", a, b)


def Iff(a, b):
    return ("iff", a, b)


def symbols(s):
    op = s[0]
    if op == "sym":
        return SortedSet([s[1]])
    elif op == "not":
        return symbols(s[1])
    elif op in ("and", "or", "imp", "iff"):
        return SortedSet(symbols(s[1]) | symbols(s[2]))
    return SortedSet()


def to_str(s):
    op = s[0]
    if op == "sym":
        return s[1]
    elif op == "not":
        return f"~{to_str(s[1])}"
    elif op == "and":
        return f"({to_str(s[1])} & {to_str(s[2])})"
    elif op == "or":
        return f"({to_str(s[1])} | {to_str(s[2])})"
    elif op == "imp":
        return f"({to_str(s[1])} => {to_str(s[2])})"
    elif op == "iff":
        return f"({to_str(s[1])} <=> {to_str(s[2])})"


s_test = Imp(And(Sym("P"), Sym("Q")), Sym("R"))
print("Symbols:", symbols(s_test))
print("String :", to_str(s_test))

Symbols: {'P', 'Q', 'R'}
String : ((P & Q) => R)


In [2]:
def pl_true(s, model):
    op = s[0]
    if op == "sym":
        return model[s[1]]
    elif op == "not":
        return not pl_true(s[1], model)
    elif op == "and":
        return pl_true(s[1], model) and pl_true(s[2], model)
    elif op == "or":
        return pl_true(s[1], model) or pl_true(s[2], model)
    elif op == "imp":
        a_val = pl_true(s[1], model)
        b_val = pl_true(s[2], model)
        return not (a_val and not b_val) 
    elif op == "iff":
        return pl_true(s[1], model) == pl_true(s[2], model)

    raise ValueError(f"Тодорхойгүй оператор: {op}")
m1 = {"P": True, "Q": False, "R": True}

sent1 = Imp(And(Sym("P"), Sym("Q")), Sym("R")) 
sent2 = Iff(Sym("P"), Sym("Q"))  
sent3 = Or(Not(Sym("P")), Sym("Q"))  

print("--- Шалгах үр дүн ---")
res1 = pl_true(sent1, m1)
res2 = pl_true(sent2, m1)
res3 = pl_true(sent3, m1)

print(f"{to_str(sent1)} утга : {res1} (Хүлээгдэж буй: True)")
print(f"{to_str(sent2)} утга       : {res2} (Хүлээгдэж буй: False)")
print(f"{to_str(sent3)} утга      : {res3} (Хүлээгдэж буй: False)")

assert res1 is True, "(P ∧ Q) ⇒ R нь True гарах ёстой!"
assert res2 is False, "P ⇔ Q нь False гарах ёстой!"
assert res3 is False, "¬P ∨ Q нь False гарах ёстой!"
print("\nБүх шалгалт амжилттай тэнцлээ!")

--- Шалгах үр дүн ---
((P & Q) => R) утга : True (Хүлээгдэж буй: True)
(P <=> Q) утга       : False (Хүлээгдэж буй: False)
(~P | Q) утга      : False (Хүлээгдэж буй: False)

Бүх шалгалт амжилттай тэнцлээ!


In [3]:
import itertools
def all_models(syms):
    sym_list = sorted(list(syms))
    for vals in itertools.product([False, True], repeat=len(sym_list)):
        yield dict(zip(sym_list, vals))


def tt_entails(kb, alpha):
    syms = symbols(kb) | symbols(alpha)
    count = 0

    for model in all_models(syms):
        count += 1
        if pl_true(kb, model):
            if not pl_true(alpha, model):
                return False, count

    return True, count


def is_valid(s):
    return all(pl_true(s, model) for model in all_models(symbols(s)))


def is_satisfiable(s):
    return any(pl_true(s, model) for model in all_models(symbols(s)))


def print_truth_table(*sentences):
    syms = set()
    for s in sentences:
        syms |= symbols(s)
    sym_list = sorted(list(syms))

    headers = [to_str(s) for s in sentences]
    col_syms = " | ".join(sym_list)
    col_expr = " | ".join(headers)
    full_header = f"{col_syms} | {col_expr}"

    print(full_header)
    print("-" * len(full_header))

    for model in all_models(sym_list):
        row_syms = " | ".join("T" if model[sym] else "F" for sym in sym_list)
        row_vals = " | ".join(
            f"{'T' if pl_true(s, model) else 'F':^{len(headers[i])}}"
            for i, s in enumerate(sentences)
        )
        print(f"{row_syms} | {row_vals}")
P, Q = Sym("P"), Sym("Q")

print("--- 1. Логик дүгнэлтүүдийг шалгах ---")

kb_mp = And(P, Imp(P, Q))
res_mp, count_mp = tt_entails(kb_mp, Q)
print(f"Modus Ponens (P ∧ (P ⇒ Q) |= Q) : {res_mp}, Шалгасан загвар: {count_mp}")
assert res_mp is True and count_mp == 4

res_inv, count_inv = tt_entails(Or(P, Q), P)
print(f"(P ∨ Q) |= P                    : {res_inv}, Шалгасан загвар: {count_inv}")
assert res_inv is False
kb_mt = And(Not(Q), Imp(P, Q))
res_mt, count_mt = tt_entails(kb_mt, Not(P))
print(f"Modus Tollens (~Q ∧ (P ⇒ Q) |= ~P): {res_mt}, Шалгасан загвар: {count_mt}")
assert res_mt is True and count_mt == 4

print("\n--- 2. Validity & Satisfiability шалгах ---")
val_res = is_valid(Or(P, Not(P)))
print(f"P ∨ ¬P is valid        : {val_res}")
assert val_res is True

sat_res = is_satisfiable(And(P, Not(P)))
print(f"P ∧ ¬P is satisfiable  : {sat_res} (unsatisfiable тул False)")
assert sat_res is False

print("\n--- 3. Де Морганы хууль (Үнэний хүснэгт) ---")
s_dm1 = Not(And(P, Q))
s_dm2 = Or(Not(P), Not(Q))
print_truth_table(s_dm1, s_dm2)

print("\nБүх шалгалт амжилттай тэнцлээ!")

--- 1. Логик дүгнэлтүүдийг шалгах ---
Modus Ponens (P ∧ (P ⇒ Q) |= Q) : True, Шалгасан загвар: 4
(P ∨ Q) |= P                    : False, Шалгасан загвар: 2
Modus Tollens (~Q ∧ (P ⇒ Q) |= ~P): True, Шалгасан загвар: 4

--- 2. Validity & Satisfiability шалгах ---
P ∨ ¬P is valid        : True
P ∧ ¬P is satisfiable  : False (unsatisfiable тул False)

--- 3. Де Морганы хууль (Үнэний хүснэгт) ---
P | Q | ~(P & Q) | (~P | ~Q)
----------------------------
F | F |    T     |     T    
F | T |    T     |     T    
T | F |    T     |     T    
T | T |    F     |     F    

Бүх шалгалт амжилттай тэнцлээ!


In [4]:
def eliminate_iff(s):
    op = s[0]
    if op == "sym":
        return s
    elif op == "not":
        return Not(eliminate_iff(s[1]))
    elif op == "iff":
        a = eliminate_iff(s[1])
        b = eliminate_iff(s[2])
        return And(Imp(a, b), Imp(b, a))
    elif op in ("and", "or", "imp"):
        return (op, eliminate_iff(s[1]), eliminate_iff(s[2]))
    return s
def eliminate_imp(s):
    op = s[0]
    if op == "sym":
        return s
    elif op == "not":
        return Not(eliminate_imp(s[1]))
    elif op == "imp":
        a = eliminate_imp(s[1])
        b = eliminate_imp(s[2])
        return Or(Not(a), b)
    elif op in ("and", "or"):
        return (op, eliminate_imp(s[1]), eliminate_imp(s[2]))
    return s
def push_not(s):
    op = s[0]
    if op == "sym":
        return s
    elif op == "not":
        sub = s[1]
        sub_op = sub[0]
        if sub_op == "sym":
            return s
        elif sub_op == "not":
            return push_not(sub[1])  
        elif sub_op == "and":
            return Or(push_not(Not(sub[1])), push_not(Not(sub[2])))
        elif sub_op == "or":
            return And(push_not(Not(sub[1])), push_not(Not(sub[2])))
    elif op == "and":
        return And(push_not(s[1]), push_not(s[2]))
    elif op == "or":
        return Or(push_not(s[1]), push_not(s[2]))
    return s
def distribute(s):
    op = s[0]
    if op in ("sym", "not"):
        return s
    elif op == "and":
        return And(distribute(s[1]), distribute(s[2]))
    elif op == "or":
        a = distribute(s[1])
        b = distribute(s[2])
        if a[0] == "and":
            return distribute(And(Or(a[1], b), Or(a[2], b)))
        elif b[0] == "and":
            return distribute(And(Or(a, b[1]), Or(a, b[2])))
        return Or(a, b)
    return s
def to_cnf(s):
    return distribute(push_not(eliminate_imp(eliminate_iff(s))))
def _get_literals(clause_tree):
    if clause_tree[0] == "or":
        return _get_literals(clause_tree[1]) | _get_literals(clause_tree[2])
    elif clause_tree[0] == "not":
        return {f"~{clause_tree[1][1]}"}
    elif clause_tree[0] == "sym":
        return {clause_tree[1]}
    raise ValueError(f"Clause доторх буруу бүтэц: {clause_tree}")
def to_clauses(s):
    cnf_s = to_cnf(s)

    def extract_conjunctions(tree):
        if tree[0] == "and":
            return extract_conjunctions(tree[1]) + extract_conjunctions(tree[2])
        return [tree]

    conjuncts = extract_conjunctions(cnf_s)
    clause_set = set()
    for c in conjuncts:
        lits = _get_literals(c)
        has_tautology = any(
            (lit[1:] if lit.startswith("~") else f"~{lit}") in lits
            for lit in lits
        )
        if not has_tautology:
            clause_set.add(frozenset(lits))
    return clause_set
print("--- 1. B <=> (A | C) clause-уудыг шалгах ---")
A, B, C = Sym("A"), Sym("B"), Sym("C")
test_sent = Iff(B, Or(A, C))
clauses = to_clauses(test_sent)
print("Үүссэн clause-ууд:")
for cl in clauses:
    print(" ", set(cl))

expected = {
    frozenset({"~B", "A", "C"}),
    frozenset({"~A", "B"}),
    frozenset({"~C", "B"}),
}
assert (
    clauses == expected
), f"Clause-ууд зөрсөн байна! Хүлээгдэж буй: {expected}, Гарсан: {clauses}"
print("Яг 3 clause зөв гарлаа!")

print("\n--- 2. CNF нь анхны өгүүлбэртэйгээ ижил үнэн утгатай эсэх (Семантик тулгалт) ---")
test_sentences = [
    test_sent,  
    Imp(And(Sym("P"), Sym("Q")), Sym("R")), 
    Not(Or(Sym("X"), And(Sym("Y"), Sym("Z")))),  
    Iff(Sym("P"), Not(Sym("Q"))), 
]

for idx, s in enumerate(test_sentences, 1):
    cnf_s = to_cnf(s)
    equiv = is_valid(Iff(s, cnf_s))
    print(f"Өгүүлбэр {idx}: {to_str(s)} <=> CNF нь valid уу? -> {equiv}")
    assert equiv, f"Өгүүлбэр {idx} дээр CNF эквивалент биш байна!"

print("\nАлхам 4 амжилттай тэнцлээ!")

--- 1. B <=> (A | C) clause-уудыг шалгах ---
Үүссэн clause-ууд:
  {'B', '~A'}
  {'~C', 'B'}
  {'~B', 'A', 'C'}
Яг 3 clause зөв гарлаа!

--- 2. CNF нь анхны өгүүлбэртэйгээ ижил үнэн утгатай эсэх (Семантик тулгалт) ---
Өгүүлбэр 1: (B <=> (A | C)) <=> CNF нь valid уу? -> True
Өгүүлбэр 2: ((P & Q) => R) <=> CNF нь valid уу? -> True
Өгүүлбэр 3: ~(X | (Y & Z)) <=> CNF нь valid уу? -> True
Өгүүлбэр 4: (P <=> ~Q) <=> CNF нь valid уу? -> True

Алхам 4 амжилттай тэнцлээ!


In [5]:
import itertools
def opposite(lit):
    return lit[1:] if lit.startswith("~") else f"~{lit}"


def resolve(ci, cj):
    resolvents = []
    for lit in ci:
        opp = opposite(lit)
        if opp in cj:
            res = (ci - {lit}) | (cj - {opp})
            if not any(opposite(l) in res for l in res):
                resolvents.append(frozenset(res))
    return resolvents


def pl_resolution(kb_clauses, alpha):
    alpha_neg_clauses = to_clauses(Not(alpha))
    clauses = set(kb_clauses) | alpha_neg_clauses
    resolvent_count = 0

    while True:
        new_clauses = set()
        pairs = list(itertools.combinations(clauses, 2))

        for ci, cj in pairs:
            res_list = resolve(ci, cj)
            for resolvent in res_list:
                resolvent_count += 1
                if len(resolvent) == 0:
                    return True, resolvent_count
                new_clauses.add(resolvent)
        if new_clauses.issubset(clauses):
            return False, resolvent_count
        clauses.update(new_clauses)
print("--- 1. resolve() функцийг шалгах ---")
res1 = resolve(frozenset({"A", "B"}), frozenset({"~A", "C"}))
print("resolve({A, B}, {~A, C}) ->", [set(r) for r in res1])
assert frozenset({"B", "C"}) in res1

res2 = resolve(frozenset({"C"}), frozenset({"~C"}))
print("resolve({C}, {~C})       ->", [set(r) for r in res2])
assert frozenset() in res2  

print("\n--- 2. pl_resolution-ийг шалгах ба tt_entails-тэй тулгах ---")
kb_clauses = {frozenset({"A", "B"}), frozenset({"~A", "C"}), frozenset({"~B", "C"})}
kb_sentence = And(Or(A, B), And(Or(Not(A), C), Or(Not(B), C)))
res_C, count_res_C = pl_resolution(kb_clauses, C)
tt_C, _ = tt_entails(kb_sentence, C)
print(f"KB |= C: Resolution={res_C} (resolvents: {count_res_C}), TT={tt_C}")
assert res_C is True and tt_C is True
res_A, count_res_A = pl_resolution(kb_clauses, A)
tt_A, _ = tt_entails(kb_sentence, A)
print(f"KB |= A: Resolution={res_A} (resolvents: {count_res_A}), TT={tt_A}")
assert res_A is False and tt_A is False

print("\nАлхам 5 амжилттай тэнцлээ!")

--- 1. resolve() функцийг шалгах ---
resolve({A, B}, {~A, C}) -> [{'B', 'C'}]
resolve({C}, {~C})       -> [set()]

--- 2. pl_resolution-ийг шалгах ба tt_entails-тэй тулгах ---
KB |= C: Resolution=True (resolvents: 21), TT=True
KB |= A: Resolution=False (resolvents: 17), TT=False

Алхам 5 амжилттай тэнцлээ!


In [6]:
def get_clause_symbols(clauses):

    syms = set()
    for c in clauses:
        for lit in c:
            syms.add(lit[1:] if lit.startswith("~") else lit)
    return syms


def find_pure_symbol(symbols, clauses, model):
    for sym in symbols:
        if sym in model:
            continue
        found_pos, found_neg = False, False
        for c in clauses:
            satisfied = any(
                (not lit.startswith("~") and model.get(lit) is True)
                or (lit.startswith("~") and model.get(lit[1:]) is False)
                for lit in c
            )
            if satisfied:
                continue

            if sym in c:
                found_pos = True
            if f"~{sym}" in c:
                found_neg = True

        if found_pos and not found_neg:
            return sym, True
        if found_neg and not found_pos:
            return sym, False
    return None, None


def find_unit_clause(clauses, model):
    for c in clauses:
        unassigned_lits = []
        is_satisfied = False
        for lit in c:
            val = model.get(lit[1:] if lit.startswith("~") else lit)
            if val is None:
                unassigned_lits.append(lit)
            elif (not lit.startswith("~") and val is True) or (
                lit.startswith("~") and val is False
            ):
                is_satisfied = True
                break

        if not is_satisfied and len(unassigned_lits) == 1:
            unit_lit = unassigned_lits[0]
            if unit_lit.startswith("~"):
                return unit_lit[1:], False
            else:
                return unit_lit, True
    return None, None


dpll_call_count = 0


def dpll(clauses, syms, model, use_pure=True, use_unit=True):
    global dpll_call_count
    dpll_call_count += 1
    all_satisfied = True
    for c in clauses:
        c_val = False
        has_unassigned = False
        for lit in c:
            val = model.get(lit[1:] if lit.startswith("~") else lit)
            if val is not None:
                lit_val = not val if lit.startswith("~") else val
                if lit_val is True:
                    c_val = True
                    break
            else:
                has_unassigned = True

        if not c_val:
            if not has_unassigned:
                return False  
            all_satisfied = False
    if all_satisfied:
        return model
    if use_pure:
        p_sym, p_val = find_pure_symbol(syms, clauses, model)
        if p_sym is not None:
            new_model = model.copy()
            new_model[p_sym] = p_val
            return dpll(clauses, syms, new_model, use_pure, use_unit)
    if use_unit:
        u_sym, u_val = find_unit_clause(clauses, model)
        if u_sym is not None:
            new_model = model.copy()
            new_model[u_sym] = u_val
            return dpll(clauses, syms, new_model, use_pure, use_unit)
    unassigned = [s for s in syms if s not in model]
    p = unassigned[0]
    m_true = model.copy()
    m_true[p] = True
    res = dpll(clauses, syms, m_true, use_pure, use_unit)
    if res:
        return res
    m_false = model.copy()
    m_false[p] = False
    return dpll(clauses, syms, m_false, use_pure, use_unit)


def dpll_satisfiable(clauses, use_pure=True, use_unit=True):
    global dpll_call_count
    dpll_call_count = 0
    syms = sorted(list(get_clause_symbols(clauses)))
    res = dpll(clauses, syms, {}, use_pure, use_unit)
    return res, dpll_call_count


print("--- 1. Satisfiable жишээг шалгах ---")
test_sat = {
    frozenset({"A"}),
    frozenset({"~A", "B"}),
    frozenset({"~B", "C"}),
    frozenset({"~C", "D", "E"}),
    frozenset({"~D", "~E"}),
}
model_found, count_sat = dpll_satisfiable(test_sat)
print("Олдсон загвар    :", model_found)
print("DPLL дуудсан тоо :", count_sat)
assert model_found is not False
assert (
    model_found["A"] is True
    and model_found["B"] is True
    and model_found["C"] is True
)
assert model_found["D"] != model_found["E"] 
print("\n--- 2. Unsatisfiable жишээг шалгах ---")
test_unsat = {
    frozenset({"A", "B"}),
    frozenset({"~A", "B"}),
    frozenset({"A", "~B"}),
    frozenset({"~A", "~B"}),
}
res_unsat, count_unsat = dpll_satisfiable(test_unsat)
print("Үр дүн           :", res_unsat, "(False байх ёстой)")
print("DPLL дуудсан тоо :", count_unsat)
assert res_unsat is False

print("\n--- 3. KB |= alpha-г DPLL-ээр шалгах ---")
entail_clauses = kb_clauses | to_clauses(Not(C))
res_entail, count_entail = dpll_satisfiable(entail_clauses)
print("CNF(KB & ~C) хангагдах эсэх :", res_entail)
print("Дүгнэлт: KB |= C нь үнэн үү? :", res_entail is False)
assert (
    res_entail is False
), "KB |= C үнэн тул KB & ~C нь unsatisfiable байх ёстой!"

print("\nАлхам 6 амжилттай тэнцлээ!")

--- 1. Satisfiable жишээг шалгах ---
Олдсон загвар    : {'A': True, 'B': True, 'C': True, 'D': True, 'E': False}
DPLL дуудсан тоо : 6

--- 2. Unsatisfiable жишээг шалгах ---
Үр дүн           : False (False байх ёстой)
DPLL дуудсан тоо : 5

--- 3. KB |= alpha-г DPLL-ээр шалгах ---
CNF(KB & ~C) хангагдах эсэх : False
Дүгнэлт: KB |= C нь үнэн үү? : True

Алхам 6 амжилттай тэнцлээ!


In [7]:
P11, P12, P21, P22, P31 = (
    Sym("P11"),
    Sym("P12"),
    Sym("P21"),
    Sym("P22"),
    Sym("P31"),
)
B11, B21 = Sym("B11"), Sym("B21")
R1 = Not(P11)  
R2 = Iff(B11, Or(P12, P21))  
R3 = Iff(
    B21, Or(P11, Or(P22, P31))
)  
R4 = Not(B11)  
R5 = B21  
def make_and_all(sentences):
    res = sentences[0]
    for s in sentences[1:]:
        res = And(res, s)
    return res


wumpus_kb = make_and_all([R1, R2, R3, R4, R5])
wumpus_kb_clauses = to_clauses(wumpus_kb)
kb_syms = symbols(wumpus_kb)
satisfying_models = [m for m in all_models(kb_syms) if pl_true(wumpus_kb, m)]
print(
    f"KB-ийг хангах загварын тоо: {len(satisfying_models)} (Хүлээгдэж буй: 3)"
)
assert len(satisfying_models) == 3, "Загварын тоо яг 3 байх ёстой!"
queries = [
    ("~P12", Not(P12), True),
    ("~P21", Not(P21), True),
    ("~P22", Not(P22), False),
    ("~P31", Not(P31), False),
    ("P22 | P31", Or(P22, P31), True),
]

print("\n" + "=" * 65)
print(
    f"{'Асуулт (α)':<14} | {'Үнэний хүснэгт':<15} | {'Резолюц':<12} | {'DPLL':<12} | {'Төлөв'}"
)
print("=" * 65)

for q_name, alpha, expected_val in queries:
    tt_res, _ = tt_entails(wumpus_kb, alpha)
    res_res, _ = pl_resolution(wumpus_kb_clauses, alpha)
    neg_alpha_clauses = to_clauses(Not(alpha))
    combined_clauses = wumpus_kb_clauses | neg_alpha_clauses
    dpll_model, _ = dpll_satisfiable(combined_clauses)
    dpll_res = dpll_model is False 
    status = (
        "OK"
        if (tt_res == res_res == dpll_res == expected_val)
        else "АЛДАА ЗӨРСӨН!"
    )

    print(
        f"{q_name:<14} | {str(tt_res):<15} | {str(res_res):<12} | {str(dpll_res):<12} | {status}"
    )

    assert (
        tt_res == res_res == dpll_res == expected_val
    ), f"{q_name} дээр аргууд зөрлөө!"

print("=" * 65)
print("Алхам 7 амжилттай тэнцлээ! Бүх 3 арга яг ижил зөв хариуг өглөө.")

KB-ийг хангах загварын тоо: 3 (Хүлээгдэж буй: 3)

Асуулт (α)     | Үнэний хүснэгт  | Резолюц      | DPLL         | Төлөв
~P12           | True            | True         | True         | OK
~P21           | True            | True         | True         | OK
~P22           | False           | False        | False        | OK
~P31           | False           | False        | False        | OK
P22 | P31      | True            | True         | True         | OK
Алхам 7 амжилттай тэнцлээ! Бүх 3 арга яг ижил зөв хариуг өглөө.


In [ ]:
import random
import time
def random_3sat(n, m, seed=1):
    random.seed(seed)
    clauses = set()
    for _ in range(m):
        vars_selected = random.sample(range(1, n + 1), 3)
        lits = []
        for v in vars_selected:
            name = f"X{v}"
            lits.append(name if random.choice([True, False]) else f"~{name}")
        clauses.add(frozenset(lits))
    return clauses
def tt_sat_count(clauses, limit=200000):
    syms = sorted(list(get_clause_symbols(clauses)))
    if 2 ** len(syms) > limit:
        return "хэт удаан"
    count = 0
    for m in all_models(syms):
        count += 1
        satisfied = True
        for c in clauses:
            if not any(
                (not lit.startswith("~") and m[lit])
                or (lit.startswith("~") and not m[lit[1:]])
                for lit in c
            ):
                satisfied = False
                break
        if satisfied:
            return count
    return count
def resolution_sat_count(clauses, limit=100000):
    current = set(clauses)
    count = 0
    while True:
        new_c = set()
        pairs = list(itertools.combinations(current, 2))
        for ci, cj in pairs:
            for r in resolve(ci, cj):
                count += 1
                if count > limit:
                    return "хэт удаан"
                if len(r) == 0:  
                    return count
                new_c.add(r)
        if new_c.issubset(current):
            return count  
        current.update(new_c)
benchmarks = [
    ("Wumpus: ~P12", "wumpus", Not(P12)),
    ("Wumpus: ~P22", "wumpus", Not(P22)),
    ("Wumpus: P22 | P31", "wumpus", Or(P22, P31)),
    ("3-SAT, n = 10", "3sat", (10, 40)),
    ("3-SAT, n = 15", "3sat", (15, 60)),
    ("3-SAT, n = 20", "3sat", (20, 80)),
]

print("Туршилтыг эхлүүлж байна (түр хүлээнэ үү)...")
print("=" * 70)
print(
    f"{'Бодлого':<20} | {'Үнэний хүснэгт (загвар)':<24} | {'Резолюц (resolvent)':<20} | {'DPLL (дуудалт)'}"
)
print("=" * 70)
for name, b_type, data in benchmarks:
    if b_type == "wumpus":
        alpha = data
        _, tt_cnt = tt_entails(wumpus_kb, alpha)
        _, res_cnt = pl_resolution(wumpus_kb_clauses, alpha)
        neg_c = to_clauses(Not(alpha))
        _, dpll_cnt = dpll_satisfiable(wumpus_kb_clauses | neg_c)

        print(f"{name:<20} | {tt_cnt:<24} | {res_cnt:<20} | {dpll_cnt}")

    elif b_type == "3sat":
        n, m = data
        sat_clauses = random_3sat(n, m, seed=1)
        tt_res = tt_sat_count(sat_clauses, limit=200000)
        res_res = resolution_sat_count(sat_clauses, limit=100000)
        _, dpll_cnt = dpll_satisfiable(sat_clauses)

        print(f"{name:<20} | {str(tt_res):<24} | {str(res_res):<20} | {dpll_cnt}")

print("=" * 70)

Туршилтыг эхлүүлж байна (түр хүлээнэ үү)...
Бодлого              | Үнэний хүснэгт (загвар)  | Резолюц (resolvent)  | DPLL (дуудалт)
Wumpus: ~P12         | 128                      | 8                    | 4
Wumpus: ~P22         | 35                       | 127                  | 7
Wumpus: P22 | P31    | 128                      | 302                  | 8
3-SAT, n = 10        | 637                      | хэт удаан            | 11
